In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True)
df = housing.frame
dftrain, dfeval = train_test_split(df[['MedInc', 'HouseAge', 'MedHouseVal']], test_size=0.2, random_state=42)

y_train = dftrain.pop('MedHouseVal')
y_eval = dfeval.pop('MedHouseVal')

plt.figure(figsize=(6, 3))
dftrain['MedInc'].hist(bins=20)
plt.title("Median Income Distribution")
plt.xlabel("Income Block Tier")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(6, 3))
plt.scatter(dftrain['HouseAge'][:500], y_train[:500], alpha=0.5, c='orange')
plt.title("House Age vs Target Value (Sample)")
plt.xlabel("House Age")
plt.ylabel("Value")
plt.show()

NUMERIC_COLUMNS = ['MedInc', 'HouseAge']
feature_columns = []
for feature_name in NUMERIC_COLUMNS:
    feature_columns.append(tf.feature_column.numeric_column(feature_name, dtype=tf.float32))

def make_input_fn(data_df, label_df, num_epochs=10, shuffle=True, batch_size=32):
    def input_function():
        ds = tf.data.Dataset.from_tensor_slices((dict(data_df), label_df))
        if shuffle:
            ds = ds.shuffle(1000)
        ds = ds.batch(batch_size).repeat(num_epochs)
        return ds
    return input_function

train_input_fn = make_input_fn(dftrain, y_train, num_epochs=20, shuffle=True)
eval_input_fn = make_input_fn(dfeval, y_eval, num_epochs=1, shuffle=False)

linear_est = tf.estimator.LinearRegressor(feature_columns=feature_columns)
linear_est.train(train_input_fn)
result = linear_est.evaluate(eval_input_fn)

print(f"Average Loss: {result['loss']:.4f}")
predictions = list(linear_est.predict(eval_input_fn))
print(f"Record 0 Predicted Property Value : {predictions[0]['predictions'][0]:.4f}")

In [ ]:
import pandas as pd
import tensorflow as tf
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
df = iris.frame
SPECIES = list(iris.target_names) 

df.columns = ['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth', 'Species']
train, test = train_test_split(df, test_size=0.2, random_state=42)
train_y = train.pop('Species')
test_y = test.pop('Species')

feature_columns = []
for key in train.keys():
    feature_columns.append(tf.feature_column.numeric_column(key=key))

def input_fn(features, labels, training=True, batch_size=32):
    dataset = tf.data.Dataset.from_tensor_slices((dict(features), labels))
    if training:
        dataset = dataset.shuffle(1000).repeat()
    return dataset.batch(batch_size)

classifier = tf.estimator.DNNClassifier(
    feature_columns=feature_columns,
    hidden_units=[30, 10],
    n_classes=3
)

classifier.train(input_fn=lambda: input_fn(train, train_y, training=True), steps=2000)
eval_result = classifier.evaluate(input_fn=lambda: input_fn(test, test_y, training=False))

print(f"Test Set Accuracy: {eval_result['accuracy']*100:.2f} %")

predict_x = {
    'SepalLength': [5.1, 5.9],
    'SepalWidth': [3.5, 3.0],
    'PetalLength': [1.4, 5.1],
    'PetalWidth': [0.2, 1.8],
}

predictions = classifier.predict(input_fn=lambda: input_fn(predict_x, labels=None, training=False))
print("\n== Custom Class Predictions ==\n")
for pred_dict in predictions:
    class_id = pred_dict['class_ids'][0]
    probability = pred_dict['probabilities'][class_id]
    print(f"Predicted Species : {SPECIES[class_id]} ( {probability*100:.1f} % Confidence )")

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

num_points = 300
num_clusters = 3
epochs = 15

X, _ = make_blobs(n_samples=num_points, centers=num_clusters, n_features=2, cluster_std=0.60, random_state=42)
data_points = tf.constant(X, dtype=tf.float32)

random_indices = tf.random.shuffle(tf.range(num_points))[:num_clusters]
centroids = tf.Variable(tf.gather(data_points, random_indices))

for epoch in range(epochs):
    distances = tf.reduce_sum(tf.square(tf.expand_dims(data_points, axis=1) - tf.expand_dims(centroids, axis=0)), axis=2)
    
    cluster_assignments = tf.argmax(distances, axis=1)
    
    new_centroids = []
    for c in range(num_clusters):
        assigned_points = tf.boolean_mask(data_points, cluster_assignments == c)
        if tf.size(assigned_points) > 0:
            new_centroids.append(tf.reduce_mean(assigned_points, axis=0))
        else:
            new_centroids.append(centroids[c])
            
    centroids.assign(tf.stack(new_centroids))

print(f"Final Group Centers : \n{centroids.numpy()}")

new_data = tf.constant([
    [2.5, 2.5],   
    [-2.0, -2.0]  
], dtype=tf.float32)
new_distances = tf.reduce_sum(tf.square(tf.expand_dims(new_data, axis=1) - tf.expand_dims(centroids, axis=0)), axis=2)

predicted_groups = tf.argmin(new_distances, axis=1)
print(f"New Point A Predicted Group : Class {predicted_groups[0].numpy()}")
print(f"New Point B Predicted Group : Class {predicted_groups[1].numpy()}")

plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], X[:, 1], c=cluster_assignments.numpy(), cmap='viridis', alpha=0.4, label='Trained Data')
plt.scatter(centroids.numpy()[:, 0], centroids.numpy()[:, 1], c='red', marker='X', s=200, label='Centroids')

plt.scatter(new_data.numpy()[:, 0], new_data.numpy()[:, 1], c='red', marker='*', s=300, edgecolor='black', label='New Predictions')

plt.title("K-Means Cluster Predictions")
plt.legend()
plt.show()

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp

tfd = tfp.distributions
true_temperatures = tf.constant([
    16.2, 15.5, 14.8, 17.1, 15.9, 4.2, 5.1, 3.8, 4.9, 14.5,
    16.0, 15.2, 17.3, 5.0, 3.9, 4.1, 5.5, 13.9, 15.1, 16.4
] * 5, dtype=tf.float32)  
eval_temperatures = tf.constant([15.8, 16.1, 4.5, 3.9, 14.7], dtype=tf.float32)

trainable_initial_logits = tf.Variable([0.0, 0.0], dtype=tf.float32)
trainable_transition_logits = tf.Variable([[0.0, 0.0], 
                                           [0.0, 0.0]], dtype=tf.float32)
trainable_means = tf.Variable([10.0, 10.0], dtype=tf.float32)
fixed_std_devs = tf.constant([3.0, 3.0], dtype=tf.float32) 

def build_hmm(num_steps):
    return tfd.HiddenMarkovModel(
        initial_distribution=tfd.Categorical(logits=trainable_initial_logits),
        transition_distribution=tfd.Categorical(logits=trainable_transition_logits),
        observation_distribution=tfd.Normal(loc=trainable_means, scale=fixed_std_devs),
        num_steps=num_steps
    )

optimizer = tf.optimizers.Adam(learning_rate=0.1)
epochs = 100
for epoch in range(epochs):
    with tf.GradientTape() as tape:
        model = build_hmm(num_steps=len(true_temperatures))
        loss = -model.log_prob(true_temperatures)
        
    gradients = tape.gradient(loss, [trainable_initial_logits, trainable_transition_logits, trainable_means])
    optimizer.apply_gradients(zip(gradients, [trainable_initial_logits, trainable_transition_logits, trainable_means]))
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1} | Training Loss : {loss.numpy():.4f}")

print("\nLearned Weather State Means | Temperatures : ", trainable_means.numpy())
eval_model = build_hmm(num_steps=len(eval_temperatures))
eval_score = eval_model.log_prob(eval_temperatures).numpy()
print(f"\nEvaluation Sequence Log Likelihood Score : {eval_score:.4f}")

test_sequence = tf.constant([16.5, 15.9, 4.1], dtype=tf.float32)
prediction_model = build_hmm(num_steps=len(test_sequence))

predicted_weather_probs = prediction_model.posterior_marginals(test_sequence).probs_parameter()

print(f"Test Sequence Input Temperatures : {test_sequence.numpy()}")
print("\nPredicted Hidden Weather State Distributions per Day :\n")
for day in range(len(test_sequence)):
    probs = predicted_weather_probs[day].numpy()
    predicted_state = tf.argmax(probs).numpy()
    print(f"Day {day+1} : State {predicted_state}")